In [ ]:
import os
from glob import glob

In [ ]:
DATA_PATH = "../data/yamaha_v0/train"


In [ ]:
image_paths = []
mask_paths = []
image_mask_pairs = glob(DATA_PATH + "/*/")
print(f"Found {len(image_mask_pairs)} image-mask pairs in {DATA_PATH}")
for image_mask in image_mask_pairs:
    image_paths.append(glob(f"{image_mask}*.jpg")[0])
    mask_paths.append(glob(f"{image_mask}*.png")[0])

In [ ]:
from collections import Counter
from PIL import Image
import numpy as np

class_counts = Counter()

for mask_path in mask_paths:
    mask = np.array(Image.open(mask_path))  # assumes HxW with class indices
    unique, counts = np.unique(mask, return_counts=True)
    class_counts.update(dict(zip(unique, counts)))

# Make sure all 9 classes are represented
total_pixels = sum(class_counts.values())
class_freqs = np.array([class_counts.get(i, 0) for i in range(9)], dtype=np.float32)

In [ ]:
import torch

epsilon = 1e-6  # To avoid division by zero
class_weights = total_pixels / (9 * (class_freqs + epsilon))
class_weights = torch.tensor(class_weights, dtype=torch.float32)

In [ ]:
class_counts

In [ ]:
class_weights